**ASSIGNMENT 07**

**NAME:** BHOSALE SRUSHTI AMAR

**CLASS:** AIDS A

**ROLL NO.:** 23107008

In [1]:
!pip install torch scikit-learn fastapi uvicorn requests pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 13.7 MB/s  0:00:54 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 12.1 MB/s  0:00:52 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 18.3 MB/s  0:00:04a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 23.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 20.3 MB/s  0:00:41 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 15.6 MB/s  0:00:12 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 23.5 MB/s  0:00:02 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 14.5 MB/s  0:00:18 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 16.8 MB/s  0:00:16 eta 

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import requests
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
conn = sqlite3.connect("cloud_data.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sensor_data(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    temperature REAL,
    vibration REAL,
    pressure REAL,
    anomaly INTEGER
)
""")

conn.commit()
conn.close()

print("Cloud database initialized")

Cloud database initialized


In [4]:
def generate_sensor_data(n=100):
    
    temperature = np.random.normal(70,10,n)
    vibration = np.random.normal(50,5,n)
    pressure = np.random.normal(30,3,n)
    
    anomaly = (
        (temperature > 85) |
        (vibration > 65) |
        (pressure > 40)
    ).astype(int)

    df = pd.DataFrame({
        "temperature":temperature,
        "vibration":vibration,
        "pressure":pressure,
        "anomaly":anomaly
    })
    
    return df

edge_data = generate_sensor_data(500)

edge_data.head()

,temperature,vibration,pressure,anomaly
0,58.539014,51.212317,33.073694,0
1,77.156137,52.612023,28.563550,0
2,74.604805,43.515301,26.789146,0
3,58.284691,50.674459,24.332127,0
4,76.923445,48.359498,25.913403,0


In [9]:
conn = sqlite3.connect("cloud_data.db")

edge_data.to_sql(
    "sensor_data",
    conn,
    if_exists="append",
    index=False
)

conn.close()

print("Edge data uploaded to cloud")

Edge data uploaded to cloud


In [11]:
conn = sqlite3.connect("cloud_data.db")
data = pd.read_sql("SELECT * FROM sensor_data", conn)
conn.close()

X = data[['temperature','vibration','pressure']]
y = data['anomaly']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train,X_test,y_train,y_test = train_test_split(
    X_scaled,y,test_size=0.2,random_state=42
)

X_train = torch.tensor(X_train,dtype=torch.float32)
y_train = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

X_test = torch.tensor(X_test,dtype=torch.float32)
y_test = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [13]:
class EdgeAIModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(3,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1),
            nn.Sigmoid()
        )
        
    def forward(self,x):
        return self.net(x)

model = EdgeAIModel()

In [15]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

epochs = 50

for epoch in range(epochs):

    preds = model(X_train)

    loss = criterion(preds,y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epoch:",epoch,"Loss:",loss.item())

Epoch: 0 Loss: 0.7073941230773926
Epoch: 10 Loss: 0.6861659288406372
Epoch: 20 Loss: 0.6654488444328308
Epoch: 30 Loss: 0.644575834274292
Epoch: 40 Loss: 0.6232016086578369


In [17]:
with torch.no_grad():
    
    preds = model(X_test)
    
    predicted = (preds > 0.5).float()
    
    accuracy = (predicted == y_test).float().mean()
    
print("Model Accuracy:",accuracy.item())

Model Accuracy: 0.9200000166893005


In [19]:
torch.save(model.state_dict(),"edge_model.pth")

print("Model exported for edge deployment")

Model exported for edge deployment


In [21]:
edge_model = EdgeAIModel()
edge_model.load_state_dict(torch.load("edge_model.pth"))

edge_model.eval()

print("Edge model loaded")

Edge model loaded


In [23]:
sample = np.array([[75,55,32]])

sample_scaled = scaler.transform(sample)

sample_tensor = torch.tensor(sample_scaled,dtype=torch.float32)

with torch.no_grad():
    
    prediction = edge_model(sample_tensor)
    
print("Anomaly Probability:",prediction.item())

Anomaly Probability: 0.40874746441841125


/home/admin1/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [25]:
if prediction.item() > 0.5:
    print("⚠️ Edge Alert: Anomaly Detected")
else:
    print("✅ System Normal")

✅ System Normal
